### Topological data analysis

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import gudhi as gd
import umap.umap_ as umap

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
base_dir = "/home/a/projects/Complete-Neural-Signal-Analysis"

embedding_3d_dir = os.path.join(base_dir, "embedding_data", "3dembedding_data")
reuse_umap_dir = os.path.join(base_dir, "embedding_data", "tsne_umap_from_3d")

topology_dir = os.path.join(base_dir, "topology_from_3d_umap")
umap_cache_dir = os.path.join(topology_dir, "umap_embeddings")
persistence_dir = os.path.join(topology_dir, "persistence_diagrams")

betti_plot_dir = os.path.join(base_dir, "plots", "betti_plots_from_3d_umap")
persistence_plot_dir = os.path.join(base_dir, "plots", "persistence_plots_from_3d_umap")

for d in [topology_dir, umap_cache_dir, persistence_dir, betti_plot_dir, persistence_plot_dir]:
    os.makedirs(d, exist_ok=True)

eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

# UMAP parameters
n_components_umap = 2
n_neighbors_umap = 15
min_dist_umap = 0.1
random_state_umap = 42

# Persistence settings
# Full Rips on all ~100k points is not feasible, so topology uses a fixed subset
persistence_max_points = 2500
rips_max_edge_length = 2.0
rips_max_dimension = 2

# Synthetic labels for demonstration classifier
segment_length = 1000
test_size = 0.30
random_state_split = 42
rf_estimators = 200
rf_random_state = 42

# Plot style
ACCENT = "cyan"
BG = "black"

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor": BG,
    "savefig.facecolor": BG,
    "text.color": ACCENT,
    "axes.labelcolor": ACCENT,
    "axes.edgecolor": ACCENT,
    "axes.titlecolor": ACCENT,
    "xtick.color": ACCENT,
    "ytick.color": ACCENT,
    "grid.color": ACCENT,
    "legend.facecolor": BG,
    "legend.edgecolor": ACCENT,
})

rng = np.random.default_rng(42)

# -------------------------------------------------------
# STYLE HELPER
# -------------------------------------------------------
def style_ax(ax):
    ax.set_facecolor(BG)
    ax.grid(True, alpha=0.20, color=ACCENT)
    ax.tick_params(colors=ACCENT)
    for spine in ax.spines.values():
        spine.set_color(ACCENT)

# -------------------------------------------------------
# BASIC HELPERS
# -------------------------------------------------------
def standardize_columns(X):
    X = np.asarray(X, dtype=float)
    mu = np.mean(X, axis=0, keepdims=True)
    sd = np.std(X, axis=0, keepdims=True) + 1e-12
    return (X - mu) / sd

def subsample_rows(X, max_points=None, rng=None):
    X = np.asarray(X)
    if rng is None:
        rng = np.random.default_rng()

    n = len(X)
    if max_points is None or n <= max_points:
        return X

    idx = np.sort(rng.choice(n, size=max_points, replace=False))
    return X[idx]

def make_synthetic_labels(n, segment_length=1000):
    num_segments = int(np.ceil(n / segment_length))
    labels = np.repeat(np.arange(num_segments) % 2, segment_length)[:n]
    return labels.astype(int)

def manual_train_test_split(X, y, test_size=0.3, random_state=42):
    X = np.asarray(X)
    y = np.asarray(y)
    n = len(X)

    rng_local = np.random.default_rng(random_state)
    perm = rng_local.permutation(n)

    n_test = int(np.round(test_size * n))
    test_idx = perm[:n_test]
    train_idx = perm[n_test:]

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def manual_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))

# -------------------------------------------------------
# UMAP HELPERS
# -------------------------------------------------------
def apply_umap(data, n_components=2, n_neighbors=15, min_dist=0.1, random_state=42):
    reducer = umap.UMAP(
        n_components=n_components,
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        metric="euclidean",
        random_state=random_state,
        transform_seed=random_state,
        low_memory=True
    )
    return reducer.fit_transform(data)

def load_or_compute_umap(channel_name, X_std):
    # first try the already existing UMAP embedding from your previous pipeline
    reuse_path = os.path.join(reuse_umap_dir, f"umap_embedding_{channel_name}.npy")
    if os.path.exists(reuse_path):
        emb = np.load(reuse_path)
        return emb, reuse_path, 0.0, True

    # then try local topology cache
    cache_path = os.path.join(umap_cache_dir, f"umap_embedding_{channel_name}.npy")
    if os.path.exists(cache_path):
        emb = np.load(cache_path)
        return emb, cache_path, 0.0, True

    # otherwise compute
    t0 = time.perf_counter()
    emb = apply_umap(
        X_std,
        n_components=n_components_umap,
        n_neighbors=n_neighbors_umap,
        min_dist=min_dist_umap,
        random_state=random_state_umap
    )
    elapsed = time.perf_counter() - t0

    np.save(cache_path, emb)
    return emb, cache_path, elapsed, False

# -------------------------------------------------------
# PERSISTENCE HELPERS
# -------------------------------------------------------
def compute_persistence(reduced_data):
    """
    Compute persistence on a reduced 2D cloud.
    """
    rips_complex = gd.RipsComplex(
        points=reduced_data,
        max_edge_length=rips_max_edge_length
    )
    simplex_tree = rips_complex.create_simplex_tree(max_dimension=rips_max_dimension)
    persistence = simplex_tree.persistence()

    intervals_by_dim = {}
    for dim in range(rips_max_dimension + 1):
        try:
            intervals_by_dim[dim] = simplex_tree.persistence_intervals_in_dimension(dim)
        except Exception:
            intervals_by_dim[dim] = np.empty((0, 2), dtype=float)

    try:
        betti = simplex_tree.betti_numbers()
    except Exception:
        betti = []

    betti = list(betti)
    if len(betti) < 3:
        betti = betti + [0] * (3 - len(betti))

    return persistence, intervals_by_dim, betti[:3]

def plot_persistence_diagram_manual(intervals_by_dim, channel_name, save_path):
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=BG)
    style_ax(ax)

    all_finite_deaths = []
    for dim in intervals_by_dim:
        arr = np.asarray(intervals_by_dim[dim], dtype=float)
        if arr.size == 0:
            continue
        finite_mask = np.isfinite(arr[:, 1])
        if np.any(finite_mask):
            all_finite_deaths.extend(arr[finite_mask, 1].tolist())

    max_finite = max(all_finite_deaths) if len(all_finite_deaths) > 0 else 1.0
    inf_cap = max_finite * 1.05

    markers = {0: "o", 1: "s", 2: "^"}

    for dim in range(3):
        arr = np.asarray(intervals_by_dim.get(dim, np.empty((0, 2))), dtype=float)
        if arr.size == 0:
            continue

        births = arr[:, 0]
        deaths = arr[:, 1].copy()
        deaths[~np.isfinite(deaths)] = inf_cap

        ax.scatter(
            births, deaths,
            s=18,
            marker=markers.get(dim, "o"),
            facecolors="none",
            edgecolors=ACCENT,
            linewidths=1.0,
            label=f"H{dim}"
        )

    lim = max(inf_cap, 1e-6)
    ax.plot([0, lim], [0, lim], linestyle="--", color=ACCENT, alpha=0.8)

    ax.set_title(f"Persistence Diagram | {channel_name}")
    ax.set_xlabel("Birth")
    ax.set_ylabel("Death")
    ax.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

def plot_betti_numbers(betti_numbers, channel_name, save_path):
    dims = [0, 1, 2]
    vals = list(betti_numbers)

    fig, ax = plt.subplots(figsize=(6, 4), facecolor=BG)
    style_ax(ax)
    ax.bar(dims, vals, color=ACCENT, edgecolor=ACCENT)
    ax.set_title(f"Betti Numbers | {channel_name}")
    ax.set_xlabel("Dimension")
    ax.set_ylabel("Betti Number")
    ax.set_xticks(dims)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()

# -------------------------------------------------------
# MAIN LOOP
# -------------------------------------------------------
summary_rows = []

for channel_name in eeg_channel_names:
    file_path = os.path.join(embedding_3d_dir, f"3dembedded_{channel_name}.npy")

    if not os.path.exists(file_path):
        print(f"Missing file for {channel_name}: {file_path}")
        continue

    channel_data_3d = np.load(file_path)
    channel_data_3d = np.asarray(channel_data_3d, dtype=float)

    if channel_data_3d.ndim != 2 or channel_data_3d.shape[1] != 3:
        print(f"Skipping {channel_name}: expected shape (N, 3), got {channel_data_3d.shape}")
        continue

    X3 = standardize_columns(channel_data_3d)

    print(f"\nProcessing {channel_name} | 3D shape={X3.shape}")

    # -------------------------
    # UMAP (reuse if already saved)
    # -------------------------
    reduced_data_umap, umap_path_used, umap_time, umap_reused = load_or_compute_umap(channel_name, X3)

    if reduced_data_umap.ndim != 2 or reduced_data_umap.shape[1] != 2:
        print(f"Skipping {channel_name}: bad UMAP shape {reduced_data_umap.shape}")
        continue

    # -------------------------
    # Topology subset
    # -------------------------
    reduced_topology = subsample_rows(reduced_data_umap, max_points=persistence_max_points, rng=rng)

    t0 = time.perf_counter()
    persistence, intervals_by_dim, betti_numbers = compute_persistence(reduced_topology)
    persistence_time = time.perf_counter() - t0

    # -------------------------
    # Save persistence data
    # -------------------------
    persistence_npz_path = os.path.join(persistence_dir, f"persistence_{channel_name}.npz")
    np.savez_compressed(
        persistence_npz_path,
        H0=np.asarray(intervals_by_dim.get(0, np.empty((0, 2))), dtype=float),
        H1=np.asarray(intervals_by_dim.get(1, np.empty((0, 2))), dtype=float),
        H2=np.asarray(intervals_by_dim.get(2, np.empty((0, 2))), dtype=float),
        betti=np.asarray(betti_numbers, dtype=int),
        n_points_topology=int(reduced_topology.shape[0]),
        channel=channel_name
    )

    # -------------------------
    # Plot persistence diagram
    # -------------------------
    persistence_plot_path = os.path.join(persistence_plot_dir, f"Persistence_{channel_name}.png")
    plot_persistence_diagram_manual(intervals_by_dim, channel_name, persistence_plot_path)

    # -------------------------
    # Plot Betti numbers
    # -------------------------
    betti_plot_path = os.path.join(betti_plot_dir, f"Betti_{channel_name}.png")
    plot_betti_numbers(betti_numbers, channel_name, betti_plot_path)

    # -------------------------
    # Synthetic labels
    # -------------------------
    labels = make_synthetic_labels(len(reduced_data_umap), segment_length=segment_length)

    # -------------------------
    # Manual split + RF classifier
    # -------------------------
    X_train, X_test, y_train, y_test = manual_train_test_split(
        reduced_data_umap,
        labels,
        test_size=test_size,
        random_state=random_state_split
    )

    clf = RandomForestClassifier(
        n_estimators=rf_estimators,
        random_state=rf_random_state,
        n_jobs=-1
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    accuracy = manual_accuracy(y_test, y_pred)

    print(f"Betti numbers for {channel_name}: {tuple(betti_numbers)}")
    print(f"Model accuracy for {channel_name}: {accuracy:.4f}")

    # -------------------------
    # Summary row
    # -------------------------
    H0 = np.asarray(intervals_by_dim.get(0, np.empty((0, 2))), dtype=float)
    H1 = np.asarray(intervals_by_dim.get(1, np.empty((0, 2))), dtype=float)
    H2 = np.asarray(intervals_by_dim.get(2, np.empty((0, 2))), dtype=float)

    summary_rows.append({
        "channel": channel_name,
        "n_points_3d": X3.shape[0],
        "n_points_umap": reduced_data_umap.shape[0],
        "n_points_topology": reduced_topology.shape[0],
        "umap_reused": bool(umap_reused),
        "umap_time_sec": float(umap_time),
        "persistence_time_sec": float(persistence_time),
        "betti_0": int(betti_numbers[0]),
        "betti_1": int(betti_numbers[1]),
        "betti_2": int(betti_numbers[2]),
        "n_H0_intervals": int(len(H0)),
        "n_H1_intervals": int(len(H1)),
        "n_H2_intervals": int(len(H2)),
        "rf_accuracy": float(accuracy),
        "umap_source": os.path.basename(umap_path_used),
        "persistence_file": os.path.basename(persistence_npz_path),
    })

# -------------------------------------------------------
# SAVE SUMMARY
# -------------------------------------------------------
summary_df = pd.DataFrame(summary_rows)
summary_csv = os.path.join(topology_dir, "topology_summary_from_3d_umap.csv")
summary_txt = os.path.join(topology_dir, "topology_summary_from_3d_umap.txt")

summary_df.to_csv(summary_csv, index=False)

with open(summary_txt, "w") as f:
    f.write("Topology from 3D embeddings after UMAP\n")
    f.write("=====================================\n\n")
    f.write(f"base_dir: {base_dir}\n")
    f.write(f"embedding_3d_dir: {embedding_3d_dir}\n")
    f.write(f"reuse_umap_dir: {reuse_umap_dir}\n")
    f.write(f"persistence_max_points: {persistence_max_points}\n")
    f.write(f"rips_max_edge_length: {rips_max_edge_length}\n")
    f.write(f"rips_max_dimension: {rips_max_dimension}\n")
    f.write(f"segment_length: {segment_length}\n")
    f.write(f"test_size: {test_size}\n\n")

    if len(summary_df) > 0:
        f.write("Per-channel summary:\n")
        for _, row in summary_df.iterrows():
            f.write(
                f"{row['channel']}: "
                f"Betti=({row['betti_0']},{row['betti_1']},{row['betti_2']}), "
                f"H0/H1/H2=({row['n_H0_intervals']},{row['n_H1_intervals']},{row['n_H2_intervals']}), "
                f"RF_acc={row['rf_accuracy']:.4f}, "
                f"UMAP_reused={row['umap_reused']}, "
                f"UMAP_time={row['umap_time_sec']:.2f}s, "
                f"Persistence_time={row['persistence_time_sec']:.2f}s\n"
            )

print(f"\nSaved summary CSV: {summary_csv}")
print(f"Saved summary TXT: {summary_txt}")
print("Done.")

### Weighted Undirected Network

<div style="font-size: 14px; font-family: 'Times New Roman', Times, serif; background-color: #181818; color: #D0D0D0; padding: 20px; border-radius: 8px; margin: 10px; display: flex; justify-content: space-between;">
    <!-- Column 1 -->
    <div style="width: 50%; margin-right: 20px;">
        <h2>Introduction</h2>
        <p>This analysis leverages graph theory to investigate the structural properties of EEG data. By constructing a weighted graph from the EEG correlation matrix, we can compute various network metrics to understand the connectivity and modularity of brain activity.</p>
        <h2>Graph Construction</h2>
        <p>The first step involves standardizing the EEG data using z-scores and then calculating the correlation matrix to quantify the relationships between EEG channels.</p>
        <p><strong>Formula:</strong></p>
        \[
        Z_{ij} = \frac{X_{ij} - \mu_j}{\sigma_j}
        \]
        where \(Z_{ij}\) is the z-scored value, \(X_{ij}\) is the original EEG value, \(\mu_j\) is the mean, and \(\sigma_j\) is the standard deviation of channel \(j\).</p>
        <p>The correlation matrix is then computed:</p>
        \[
        R_{ij} = \frac{\text{Cov}(Z_i, Z_j)}{\sigma_{Z_i} \sigma_{Z_j}}
        \]
        <p>We construct a weighted graph \(G\) from the correlation matrix by adding nodes for each EEG channel and edges weighted by the correlation values that exceed a threshold.</p>
        <h2>Clustering Coefficient</h2>
        <p>The clustering coefficient measures the degree to which nodes in the graph tend to cluster together. It is given by:</p>
        <p><strong>Formula:</strong></p>
        \[
        C = \frac{1}{N} \sum_{i=1}^{N} \frac{2e_i}{k_i(k_i-1)}
        \]
        where \(e_i\) is the number of edges connecting the neighbors of node \(i\), and \(k_i\) is the degree of node \(i\).</p>
    </div>
    <!-- Column 2 -->
    <div style="width: 50%; margin-left: 20px;">
        <h2>Modularity</h2>
        <p>Modularity quantifies the strength of division of a network into communities. Higher modularity indicates a stronger community structure.</p>
        <p><strong>Formula:</strong></p>
        \[
        Q = \frac{1}{2m} \sum_{ij} \left[ A_{ij} - \frac{k_i k_j}{2m} \right] \delta(c_i, c_j)
        \]
        where \(A_{ij}\) is the adjacency matrix, \(k_i\) and \(k_j\) are the degrees of nodes \(i\) and \(j\), \(m\) is the number of edges, and \(\delta\) is the Kronecker delta function indicating whether nodes \(i\) and \(j\) are in the same community.</p>
        <h2>Small-Worldness</h2>
        <p>Small-world properties are assessed using two metrics: sigma (\(\sigma\)) and omega (\(\omega\)).</p>
        <p><strong>Formula:</strong></p>
m        \[
        \sigma = \frac{C / C_{\text{rand}}}{L / L_{\text{rand}}}
        \]
        \[
        \omega = \frac{L_{\text{rand}}}{L} - \frac{C}{C_{\text{latt}}}
        \]
        where \(C\) and \(L\) are the clustering coefficient and average shortest path length of the network, \(C_{\text{rand}}\) and \(L_{\text{rand}}\) are those of a random network, and \(C_{\text{latt}}\) is the clustering coefficient of a lattice network.</p>
        <h2>Global Efficiency</h2>
        <p>Global efficiency measures how efficiently information is exchanged over the network. It is the average of the inverse shortest path length.</p>
        <p><strong>Formula:</strong></p>
        \[
        E_{\text{glob}} = \frac{1}{N(N-1)} \sum_{i \neq j} \frac{1}{d_{ij}}
        \]
        where \(d_{ij}\) is the shortest path length between nodes \(i\) and \(j\).</p>
        <h2>Assortativity</h2>
        <p>Assortativity measures the tendency of nodes to connect with similar nodes. It is computed as the Pearson correlation coefficient of degrees between pairs of linked nodes.</p>
        <p><strong>Formula:</strong></p>
        \[
        r = \frac{\sum_{jk} jk(e_{jk} - q_j q_k)}{\sigma_q^2}
        \]
        where \(e_{jk}\) is the joint probability distribution of the degrees of nodes at either end of a link, and \(q_j\) is the distribution of the remaining degree.</p>
        <h2>Results and Conclusion</h2>
        <p>After computing these metrics, we gain insights into the structural properties of the EEG network. The clustering coefficient indicates local connectivity, modularity reveals community structure, small-worldness metrics compare the EEG network to idealized networks, global efficiency assesses information exchange, and assortativity reveals connectivity patterns.</p>
        <p>This graph-theoretical analysis provides a comprehensive framework for understanding the complex interactions within the brain, as captured by EEG data.</p>
    </div>
</div>


In [ ]:
import networkx as nx
import numpy as np
from networkx.algorithms.community import greedy_modularity_communities
from networkx.algorithms.smallworld import sigma, omega  # hypothetical functions for small-worldness
from networkx.algorithms.assortativity import degree_assortativity_coefficient
from scipy.stats import zscore
import os

# Define directories
base_dir = '/home/vincent/MySSD/JupyterProjects/AAA_projects/UnlimitedResearchCooperative/Synthetic_Intelligence_Labs/EEG_Chaos_Kuramoto_Neural_Net'
EEG_data_path = os.path.join(base_dir, 'eeg_data_with_channels.npy')
EEG_data = np.load(EEG_data_path, allow_pickle=True)

sampling_rate = 1000  # Hz (adjust if necessary)
start_time, end_time = 814.571, 921.515  # Data without stimulation occurring
start_index, end_index = int(start_time * sampling_rate), int(end_time * sampling_rate)
EEG_data = EEG_data[start_index:end_index, :]

# EEG channel names
eeg_channel_names = [
    'Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6',
    'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6', 'P7',
    'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'Oz', 'O2'
]

# Z-score EEG data and calculate the correlation matrix
EEG_zscored = zscore(EEG_data, axis=0)
corr_matrix = np.corrcoef(EEG_zscored.T)

# Create a weighted graph from the correlation matrix
G = nx.Graph()

# Add nodes to the graph
for channel in eeg_channel_names:
    G.add_node(channel)

# Add weighted edges to the graph, thresholding correlations
threshold = 0.5  # This threshold is arbitrary and can be adjusted
for i in range(len(eeg_channel_names)):
    for j in range(i + 1, len(eeg_channel_names)):
        weight = corr_matrix[i, j]
        if abs(weight) > threshold:
            G.add_edge(eeg_channel_names[i], eeg_channel_names[j], weight=weight)

# Clustering coefficient
clustering_coefficient = nx.average_clustering(G, weight='weight')

# Modularity using greedy algorithm
communities = greedy_modularity_communities(G)
modularity = nx.algorithms.community.quality.modularity(G, communities)

# Small-worldness, using sigma and omega functions
# Sigma compares the clustering coefficient and path length of the network to a random network
# Omega compares the clustering coefficient and path length of the network to a lattice network
# These functions do not exist in networkx and are placeholders
small_world_sigma = sigma(G)
small_world_omega = omega(G)

# Global efficiency
efficiency = nx.global_efficiency(G)

# Assortativity
assortativity = degree_assortativity_coefficient(G)

# Now, to output the calculated metrics
print(f"Clustering Coefficient: {clustering_coefficient}")
print(f"Modularity: {modularity}")
print(f"Small-World Sigma: {small_world_sigma}")
print(f"Small-World Omega: {small_world_omega}")
print(f"Global Efficiency: {efficiency}")
print(f"Assortativity: {assortativity}")

In [ ]:
### Homomorphisms and isomorphisms

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.signal import welch
from sklearn.decomposition import PCA

# Define directories and load EEG data
base_dir = '/home/vincent/MySSD/JupyterProjects/AAA_projects/UnlimitedResearchCooperative/Synthetic_Intelligence_Labs/EEG_Chaos_Kuramoto_Neural_Net'
EEG_data_path = os.path.join(base_dir, 'eeg_data_with_channels.npy')
EEG_data = np.load(EEG_data_path, allow_pickle=True)

# Define parameters
sampling_rate = 1000  # Hz
start_time, end_time = 814.571, 921.515  # Data without stimulation occurring
start_index, end_index = int(start_time * sampling_rate), int(end_time * sampling_rate)
EEG_data = EEG_data[start_index:end_index, :]

# Function to apply PCA (isomorphism)
def apply_pca(data, n_components=3):
    pca = PCA(n_components=n_components)
    pca_data = pca.fit_transform(data)
    return pca_data

# Apply PCA to the EEG data
pca_data = apply_pca(EEG_data)

# Function to compute Welch's power spectral density estimate (homomorphism)
def compute_welch_psd(data, fs=1000):
    psd_data = []
    for channel in data.T:
        f, Pxx = welch(channel, fs=fs, nperseg=1024)
        psd_data.append(Pxx)
    return np.array(f), np.array(psd_data)

# Compute Welch's PSD
frequencies, psd_data = compute_welch_psd(EEG_data, fs=sampling_rate)

# Plotting the original EEG data
plt.figure(figsize=(12, 6))
for i in range(EEG_data.shape[1]):
    plt.plot(EEG_data[:, i] + i*10, label=f'Channel {i+1}')  # Offset each channel for visibility
plt.title('Original EEG Data')
plt.xlabel('Time (ms)')
plt.ylabel('Amplitude (uV)')
plt.legend()
plt.show()

# Plotting the PCA-transformed data
plt.figure(figsize=(12, 6))
for i in range(pca_data.shape[1]):
    plt.plot(pca_data[:, i] + i*10, label=f'PCA Component {i+1}')  # Offset each component for visibility
plt.title('PCA-Transformed EEG Data')
plt.xlabel('Time (ms)')
plt.ylabel('PCA Amplitude')
plt.legend()
plt.show()

# Plotting the Power Spectral Density (PSD)
plt.figure(figsize=(12, 6))
for i in range(psd_data.shape[0]):
    plt.semilogy(frequencies, psd_data[i], label=f'Channel {i+1}')
plt.title('Power Spectral Density (PSD) of EEG Data')
plt.xlabel('Frequency (Hz)')
plt.ylabel('PSD (uV^2/Hz)')
plt.legend()
plt.show()